# GreenCampus AI – Campus Sustainability Data Generation

## 1. Project Overview

GreenCampus AI is an AI-powered campus sustainability monitoring system
designed to monitor and analyze electricity, water, and waste consumption.

The project is aligned with SDG 12 – Responsible Consumption and Production.

This notebook generates a simulated campus sustainability dataset for
testing the proposed AI system.

In [ ]:
import pandas as pd
import numpy as np

## 2. Dataset Objectives

The dataset is designed to represent:

- Electricity consumption
- Water consumption
- Waste generation
- Campus occupancy
- Temperature
- Working days
- Event days
- Different campus buildings

The dataset is simulated and does not represent actual college data.

#### Create dates

In [ ]:
dates = pd.date_range(start = "2025-07-01", end = "2026-06-30", freq = "D")

## 3. Campus Buildings

The simulated campus contains five locations:

- Block A – Academic
- Block B – Academic
- Block C – Laboratory
- Library – Library
- Hostel – Hostel

In [ ]:
buildings = {"Block A": {
    "Building_Type": "Academic", "Base_Occupancy": 420
},
    "Block B": {
    "Building_Type": "Academic", "Base_Occupancy": 310
},
    "Block C": {
    "Building_Type": "Laboratory", "Base_Occupancy": 180
},
    "Library": {
    "Building_Type": "Library", "Base_Occupancy": 250
},
    "Hostel": {
    "Building_Type": "Hostel", "Base_Occupancy": 500
}
            }

## 4. Data Generation

In [ ]:
records = []

for date in dates:
    #days information
    day_name = date.day_name()

    #Monday-Friday = working day
    working_day = day_name not in ["Saturday", "Sunday"]

    #Randomly create some campus event days
    event_day = np.random.choice(
        ["Yes", "No"],
        p = [0.05, 0.95]
    )

    #simulated temperature
    month = date.month

    if month in [12, 1, 2]:
        temperature = np.random.normal(22, 3)
    elif month in [3, 4, 5]:
        temperature = np.random.normal(32, 4)
    else:
        temperature = np.random.normal(29, 3)

    temperature = round(max(15, temperature), 1)

    for building, info in buildings.items():

        # Occupancy
        if working_day:
            occupancy = np.random.normal(
                info["Base_Occupancy"], info["Base_Occupancy"] * 0.08
            )
        else:
            occupancy = np.random.normal(
                info["Base_Occupancy"] * 0.25, info["Base_Occupancy"] * 0.08
            )
        # Event increases occupancy
        if event_day == "Yes":
            occupancy *= 1.15
        occupancy = int(max(10, occupancy))

        #Electricity consumption
        electricity = (
            info["Base_Occupancy"] * 2.5 + occupancy * 0.8 + max(temperature - 25, 0) * 25 + np.random.normal(0, 80)
        )

        #Lower consumption on non-working days
        if not working_day:
            electricity *= 0.65

        #Water consumption
        water = (
            occupancy * 18 + np.random.normal(0, 500)
        )

        if not working_day:
            water *= 0.65

        #Waste Generation
        waste = (
            occupancy * 0.10 + np.random.normal(0, 3)
        )
        if event_day == "Yes":
            waste *= 1.30

        #Keep values positive
        electricity = max(100, electricity)
        water = max(500, water)
        waste = max(2, waste)

        records.append([
            date, building, info["Building_Type"], occupancy, round(electricity, 2), round(water, 2), round(waste, 2), temperature,
            "Yes" if working_day else "No", event_day
        ])


#### Create dataframes

In [ ]:
columns = ["Date", "Building", "Building_Type", "Occupancy", "Electricity_kWh", "Water_Litres", "Waste_kg", "Temperature_C", "Working_Day", "Event_Day"]
df = pd.DataFrame(records, columns=columns)

#### Add some intentional anomalies

In [ ]:
# Electricity anomaly
df.loc[(df["Date"] == "2026-02-15") & (df["Building"] == "Block A"), "Electricity_kWh"] *= 2.2

# Water anomaly
df.loc[(df["Date"] == "2026-04-10") & (df["Building"] == "Block B"), "Water_Litres"] *= 2.3

# Waste anomaly
df.loc[(df["Date"] == "2026-05-20") & (df["Building"] == "Library"), "Waste_kg"] *= 2.5

#Round values again
df["Electricity_kWh"] = df["Electricity_kWh"].round(2)
df["Water_Litres"] = df["Water_Litres"].round(2)
df["Waste_kg"] = df["Waste_kg"].round(2)

#### Save csv

In [ ]:
df.to_csv("campus_sustainability_data.csv", index=False)

## 5. Dataset Verification

The generated dataset is checked for its shape, building distribution,
basic statistics, and data quality.

In [ ]:
print("Dataset created successfully!")
print("Number of rows: ", len(df))
print("Number of columns: ", len(df.columns))

print("\nFirst 5 rows of dataset:")
display(df.head())

print("\nDataset information:")
display(df.info())

print("\nMissing values:")
display(df.isnull().sum())

In [ ]:
df.shape

In [ ]:
df.head(5)

In [ ]:
df["Building"].value_counts()

In [ ]:
df.describe